# You Can Flip Any Election by Moving People Around: 2012

# Data Source

In [1]:
import pandas as pd

In [2]:
# data source: 2012 election results, wikipedia
url = "https://en.wikipedia.org/wiki/2012_United_States_presidential_election"

In [21]:
import requests

header = {
  "User-Agent": "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/50.0.2661.75 Safari/537.36",
  "X-Requested-With": "XMLHttpRequest"
}

r = requests.get(url, headers=header)

tables = pd.read_html(r.text) # array of all dataframes

C:\Users\yasha\AppData\Local\Temp\ipykernel_20288\3458011634.py:10: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  tables = pd.read_html(r.text) # array of all dataframes


In [23]:
raw_df = tables[22]
raw_df.head()

State or district Obama/Biden Democratic             Romney/Ryan Republican  \
  State or district                      #       %  EV                      #   
0           Alabama                 795696  38.36%   –                1255925   
1            Alaska                 122640  40.81%   –                 164676   
2           Arizona                1025232  44.59%   –                1233654   
3          Arkansas                 394409  36.88%   –                 647744   
4        California                7854285  60.24%  55                4839958   

              Johnson/Gray Libertarian            ... Stein/Honkala Green     \
        %  EV                        #      % EV  ...                   % EV   
0  60.55%   9                    12328  0.59%  –  ...               0.16%  –   
1  54.80%   3                     7392  2.46%  –  ...               0.97%  –   
2  53.65%  11                    32100  1.39%  –  ...               0.34%  –   
3  60.57%   6                    16276  1.52%  –  ...               0.87%  –   
4  37.12%   –                   143221  1.10%  –  ...               0.66%  –   

   Others              Margin          Margin Swing[a] Total votes  \
        #      % EV         #        %               % Total votes   
0    6992  0.33%  –  −460,229  −22.19%          −0.61%     2074338   
1    2870  0.96%  –   −42,036  −13.99%           7.55%      300495   
2     452  0.02%  –  −208,422   −9.06%          −0.54%     2299254   
3    1734  0.16%  –  −253,335  −23.69%          −3.84%     1069468   
4  115445  0.88%  –   3014327   23.12%          −0.94%    13038547   

  Unnamed: 20_level_0  
  Unnamed: 20_level_1  
0                  AL  
1                  AK  
2                  AZ  
3                  AR  
4                  CA  

[5 rows x 21 columns]

# Data Transformation

In [48]:
df = raw_df.copy(deep=True)

# remove the [] artifacts
import re

# Example column name
col = "State or district"
states = df[col][col]


# Remove square brackets and anything inside them
states = (
    states
    .str.replace(r"\[.*?\]", "", regex=True)          # remove [..]
    .str.replace(r"Tooltip.*", "", regex=True)        # remove Tooltip + following text
    .str.replace("†", "", regex=False)                # remove dagger symbol
    .str.strip()                                      # remove leading/trailing spaces
)


df.insert(1, "States", states)
df=df.drop("State or district", axis=1)

# remove unneeded columns, row, renamed for ease
df=df.drop(["Johnson/Gray Libertarian","Stein/Honkala Green","Others","Margin Swing[a]","Unnamed: 20_level_0"],level=0,axis=1)
df=df.drop(57)

ogwin = df.columns[1][0]
oglos = df.columns[4][0]
# df = df.rename(columns={"Obama/Biden Democratic": ogwin, 'Romney/Ryan Republican': oglos})
df.head()

C:\Users\yasha\AppData\Local\Temp\ipykernel_20288\3267666995.py:22: PerformanceWarning: dropping on a non-lexsorted multi-index without a level parameter may impact performance.
  df=df.drop("State or district", axis=1)


States Obama/Biden Democratic             Romney/Ryan Republican  \
                                   #       %  EV                      #   
0     Alabama                 795696  38.36%   –                1255925   
1      Alaska                 122640  40.81%   –                 164676   
2     Arizona                1025232  44.59%   –                1233654   
3    Arkansas                 394409  36.88%   –                 647744   
4  California                7854285  60.24%  55                4839958   

                 Margin          Total votes  
        %  EV         #        % Total votes  
0  60.55%   9  −460,229  −22.19%     2074338  
1  54.80%   3   −42,036  −13.99%      300495  
2  53.65%  11  −208,422   −9.06%     2299254  
3  60.57%   6  −253,335  −23.69%     1069468  
4  37.12%   –   3014327   23.12%    13038547

In [51]:
# convert votes, %, EV from str to int
# apparently the minus signs are not the correct symbol, which is tripping up the str to int coversion, so we have to fix those dashes

num_df=df.copy(deep=True) # create new dataframe for simplicity

DASHES = "\u2212\u2012\u2013\u2014\u2015"  # minus, figure dash, en dash, em dash, horiz bar
num_cols=[ogwin,oglos,"Margin"]
for col in num_cols:
  s=df[col]['#']
  s = (
    s.str.strip()
    # Normalize any Unicode dash to ASCII hyphen-minus
    .str.replace(f"[{DASHES}]", "-", regex=True)
    # Remove commas used as thousands separators
    .str.replace(",", "", regex=False)
  )
  nums = pd.to_numeric(s, errors="coerce")

  # Convert to nullable integer type
  num_df.loc[:,(col,'#')] = nums.astype("Int64") # truncates toward 0 (e.g., -1.9 -> -1)

  # %
  p=df[col]['%']
  p = (
    p.str.strip()
    # Normalize any Unicode dash to ASCII hyphen-minus
    .str.replace(f"[{DASHES}]", "-", regex=True)
    # Remove %
    .str.replace("%", "", regex=False)
  )
  nums = pd.to_numeric(p, errors="coerce")

  # Convert to nullable integer type
  num_df.loc[:,(col,'%')] = nums.astype("Float64") # truncates toward 0 (e.g., -1.9 -> -1)


  #  EV
  if col!='Margin': # margin doesnt have an EV col
    e=df[col]['EV']
    e=e.str.replace(f"[{DASHES}]", "0", regex=True) # the "nulls" are just dashes so we can replace them with 0s
    nums = pd.to_numeric(e, errors="coerce")
    num_df.loc[:,(col,'EV')] = nums.astype("Int64")

#convert Total
num_df.rename(columns={"Total votes":"Total"},inplace=True)
col="Total votes"
s=df[col][col]
nums = pd.to_numeric(s, errors="coerce")
num_df['Total Votes'] = nums.astype("Int64")
num_df=num_df.drop("Total", axis=1) #drop the old total

num_df

C:\Users\yasha\AppData\Local\Temp\ipykernel_20288\2611817836.py:50: PerformanceWarning: dropping on a non-lexsorted multi-index without a level parameter may impact performance.
  num_df=num_df.drop("Total", axis=1) #drop the old total


States Obama/Biden Democratic              \
                                              #      %   EV   
0                Alabama                 795696  38.36    0   
1                 Alaska                 122640  40.81    0   
2                Arizona                1025232  44.59    0   
3               Arkansas                 394409  36.88    0   
4             California                7854285  60.24   55   
5               Colorado                1323102  51.49    9   
6            Connecticut                 905083  58.06    7   
7               Delaware                 242584  58.61    3   
8   District of Columbia                 267070  90.91    3   
9                Florida                4237756  50.01   29   
10               Georgia                1773827  45.48    0   
11                Hawaii                 306658  70.55    4   
12                 Idaho                 212787   32.4    0   
13              Illinois                3019512   57.6   20   
14               Indiana                1152887  43.93    0   
15                  Iowa                 822544  51.99    6   
16                Kansas                 440726  37.99    0   
17              Kentucky                 679370   37.8    0   
18             Louisiana                 809141  40.58    0   
19                 Maine                 401306  56.27    2   
20                  ME-1                 223035  59.57    1   
21                  ME-2                 177998  52.94    1   
22              Maryland                1677844  61.97   10   
23         Massachusetts                1921290  60.65   11   
24              Michigan                2564569  54.21   16   
25             Minnesota                1546167  52.65   10   
26           Mississippi                 562949  43.79    0   
27              Missouri                1223796  44.38    0   
28               Montana                 201839   41.7    0   
29              Nebraska                 302081  38.03    0   
30                  NE-1                 108082  40.83    0   
31                  NE-2                 121889   45.7    0   
32                  NE-3                  72110  27.82    0   
33                Nevada                 531373  52.36    6   
34         New Hampshire                 369561  51.98    4   
35            New Jersey                2125101  58.38   14   
36            New Mexico                 415335  52.99    5   
37              New York                4485741  63.35   29   
38        North Carolina                2178391  48.35    0   
39          North Dakota                 124827  38.69    0   
40                  Ohio                2827709  50.67   18   
41              Oklahoma                 443547  33.23    0   
42                Oregon                 970488  54.24    7   
43          Pennsylvania                2990274  51.97   20   
44          Rhode Island                 279677   62.7    4   
45        South Carolina                 865941  44.09    0   
46          South Dakota                 145039  39.87    0   
47             Tennessee                 960709  39.08    0   
48                 Texas                3308124  41.38    0   
49                  Utah                 251813  24.75    0   
50               Vermont                 199239  66.57    3   
51              Virginia                1971820  51.16   13   
52            Washington                1755396  56.16   12   
53         West Virginia                 238269  35.54    0   
54             Wisconsin                1620985  52.83   10   
55               Wyoming                  69286  27.82    0   
56                 Total               65915795  51.06  332   

   Romney/Ryan Republican                Margin        Total Votes  
                        #      %   EV         #      %              
0                 1255925  60.55    9   -460229 -22.19     2074338  
1                  164676   54.8    3    -42036 -13.99      300495  
2                 123365

In [52]:
# popular vote
national=num_df.iloc[56]
num_df=num_df.drop(56)
national

States                            Total
Obama/Biden Democratic  #      65915795
                        %         51.06
                        EV          332
Romney/Ryan Republican  #      60933504
                        %          47.2
                        EV          206
Margin                  #       4982291
                        %          3.86
Total Votes                   129085410
Name: 56, dtype: object

In [54]:
# VPP

num_df['VPP']=num_df["Margin"]['#']/(num_df[oglos]['EV']+num_df[ogwin]['EV'])
num_df['VPP']=num_df['VPP'].astype(float).round(2)

# Flipping Algorithm
1.   Calculate Minimum EV needed to flip
2.   Split into winner and loser dfs

In [55]:
# Minimum EV
import math
nat_total_EV=national[(ogwin,"EV")]+national[(oglos,"EV")]
if nat_total_EV%2==0:
  threshold_EV=int(nat_total_EV/2)+1
else:
  threshold_EV=math.ceil(nat_total_EV/2)
minimum_EV=threshold_EV-national[(oglos,"EV")] #the minimum EV to flip to flip the entire election
minimum_EV

64

In [69]:
losers_df=num_df[num_df['Margin']['#']<0]
losers_df=losers_df.sort_values(by=[('Margin',"#")])
losers_df.reset_index(inplace=True) #reset index to put low VPP at top
losers_df.drop("index", axis=1, inplace=True) #drop original mixed up index
losers_df.head()

C:\Users\yasha\AppData\Local\Temp\ipykernel_20288\2698913870.py:4: PerformanceWarning: dropping on a non-lexsorted multi-index without a level parameter may impact performance.
  losers_df.drop("index", axis=1, inplace=True) #drop original mixed up index


States Obama/Biden Democratic           Romney/Ryan Republican         \
                                  #      % EV                      #      %   
0      Texas                3308124  41.38  0                4569843  57.17   
1  Tennessee                 960709  39.08  0                1462330  59.48   
2       Utah                 251813  24.75  0                 740600  72.79   
3    Alabama                 795696  38.36  0                1255925  60.55   
4   Oklahoma                 443547  33.23  0                 891325  66.77   

         Margin        Total Votes       VPP  
   EV         #      %                        
0  38  -1261719 -15.79     7993851 -33203.13  
1  11   -501621  -20.4     2458577 -45601.91  
2   6   -488787 -48.04     1017440 -81464.50  
3   9   -460229 -22.19     2074338 -51136.56  
4   7   -447778 -33.54     1334872 -63968.29

In [72]:
winners_df=num_df[num_df['Margin']['#']>0]
winners_df=winners_df.sort_values(by='VPP')
winners_df.reset_index(inplace=True) #reset index to put low VPP at top
winners_df.drop("index", axis=1, inplace=True) #drop original mixed up index
winners_df.head()

C:\Users\yasha\AppData\Local\Temp\ipykernel_20288\1025382241.py:4: PerformanceWarning: dropping on a non-lexsorted multi-index without a level parameter may impact performance.
  winners_df.drop("index", axis=1, inplace=True) #drop original mixed up index


States Obama/Biden Democratic            Romney/Ryan Republican  \
                                      #      %  EV                      #   
0        Florida                4237756  50.01  29                4163447   
1           Ohio                2827709  50.67  18                2661437   
2  New Hampshire                 369561  51.98   4                 329918   
3         Nevada                 531373  52.36   6                 463567   
4       Virginia                1971820  51.16  13                1822522   

             Margin       Total Votes       VPP  
       % EV       #     %                        
0  49.13  0   74309  0.88     8474179   2562.38  
1  47.69  0  166272  2.98     5580847   9237.33  
2   46.4  0   39643  5.58      710972   9910.75  
3  45.68  0   67806  6.68     1014918  11301.00  
4  47.28  0  149298  3.88     3854489  11484.46

# Overflow Algorithm: Minimum EV
In 2012, Romney's biggest surplus was in Texas.

In [64]:
mow_df=losers_df.copy(deep=True)

mol_df=winners_df.copy(deep=True)

sources=1 #number of source states
flipped_EV=0
total_voters_moved=0 #the number of voters moved overall
state_voters_moved=0 #the number of voters moved into each state to flip it
mini_margin=1000  #our arbitrary minimum acceptable margin of statewide victory

i=0 # source state index
j=0 # target state index
pover=0 # if a state's overflow is less than the margin, carry over to the next source state's overflow

source_state=mow_df.iloc[i] #the state we are moving voters from
target_state=mol_df.iloc[j] #the state we are moving voters to
overflow=source_state.loc[("Margin","#")]*(-1)-mini_margin+pover # the number of voters we have to move from the source state, plus any leftover voters we had to carry over from the previous source state


while i<sources and j<len(mol_df): # while we still have source states and target states to move voters from/to

  if overflow>(target_state.loc[("Margin","#")]+mini_margin): #if we have enough voters in the source state to flip the target state with a mini_margin, move the number of voters needed to flip the target state with a mini_margin
    pover=0 # reset pover since we have enough voters to flip the target state
    
    state_voters_moved=target_state.loc[("Margin","#")]+mini_margin
    # change votes
    mol_df.loc[j,(oglos,"#")]=int(mol_df.loc[j,(oglos,"#")])+state_voters_moved
    mol_df.loc[j,"Total Votes"]=int(mol_df["Total Votes"].iloc[j])+state_voters_moved
    mol_df.loc[j,("Margin","#")]=mini_margin*(-1)

    # change EV
    flipped_EV=flipped_EV+mol_df.loc[j,(ogwin,"EV")]
    mol_df.loc[j,(oglos,"EV")]=mol_df.loc[j,(ogwin,"EV")]
    mol_df.loc[j,(ogwin,"EV")]=0
    
    # change overflow
    overflow=overflow-state_voters_moved
    total_voters_moved=total_voters_moved+state_voters_moved # update total voters moved
    
    print(f"Flipped {target_state.loc['States'].to_string()[4:]} by moving {state_voters_moved} voters from {source_state.loc['States'].to_string()[4:]}. Total voters moved: {total_voters_moved}. Flipped EV: {flipped_EV}. Overflow: {overflow}.")
    if flipped_EV>=minimum_EV:
      print(f"Flipped election! Flipped EV: {flipped_EV}. Minimum EV needed: {minimum_EV}. Total voters moved: {total_voters_moved}.")
      break
    
    # move on to the next target state
    j=j+1
    target_state=mol_df.iloc[j] 

  else: # store overflow in pover and move on to the next source state
    print(f"Not enough voters in {source_state.loc['States'].to_string()[4:]} to flip {target_state.loc['States'].to_string()[4:]}. Overflow: {overflow}. Moving on to the next source state.")
    pover=overflow
    i+=1 
    source_state=mow_df.iloc[i] if i<sources else None
    overflow=source_state.loc[("Margin","#")]*(-1)-mini_margin+pover if i<sources else None # update overflow for the new source state




# mol_df

Flipped Florida by moving 75309 voters from Texas. Total voters moved: 75309. Flipped EV: 29. Overflow: 1185410.
Flipped Ohio by moving 167272 voters from Texas. Total voters moved: 242581. Flipped EV: 47. Overflow: 1018138.
Flipped New Hampshire by moving 40643 voters from Texas. Total voters moved: 283224. Flipped EV: 51. Overflow: 977495.
Flipped Nevada by moving 68806 voters from Texas. Total voters moved: 352030. Flipped EV: 57. Overflow: 908689.
Flipped Virginia by moving 150298 voters from Texas. Total voters moved: 502328. Flipped EV: 70. Overflow: 758391.
Flipped election! Flipped EV: 70. Minimum EV needed: 64. Total voters moved: 502328.


In [65]:
# update the source states
mow_df=losers_df.copy(deep=True)

i=0
reimburse = total_voters_moved
while i<sources:
    if reimburse>mow_df.loc[i,("Margin","#")]*(-1)-mini_margin:
        state_voters_removed=mow_df.loc[i,("Margin","#")]*(-1)-mini_margin

        # change votes
        mow_df.loc[i,(oglos,"#")]=int(mow_df.loc[i,(oglos,"#")])-state_voters_removed
        mow_df.loc[i,"Total Votes"]=int(mow_df["Total Votes"].iloc[i])-state_voters_removed
        mow_df.loc[i,("Margin","#")]=mini_margin*(-1)

        reimburse-=state_voters_removed
        print(f"Removed {state_voters_removed} voters from {mow_df.loc[i,'States'].to_string()[4:]}. Remaining voters to reimburse: {reimburse}.")
    else:
        state_voters_removed=reimburse

        # change votes
        mow_df.loc[i,(oglos,"#")]=int(mow_df.loc[i,(oglos,"#")])-state_voters_removed
        mow_df.loc[i,"Total Votes"]=int(mow_df["Total Votes"].iloc[i])-state_voters_removed
        mow_df.loc[i,("Margin","#")]=mow_df.loc[i,("Margin","#")]+state_voters_removed

        reimburse-=state_voters_removed
        print(f"Removed {state_voters_removed} voters from {mow_df.loc[i,'States'].to_string()[4:]}. All voters reimbursed.")
    i+=1
        
# mow_df

Removed 502328 voters from Texas. All voters reimbursed.


In [ ]:
# concatenate the winner and loser
mfinal_df=pd.concat([mol_df,mow_df]).sort_values(by=[("Margin","Votes")])

#update percentages
mfinal_df.loc[:,(oglos,"%")]=(mfinal_df.loc[:,(oglos,"Votes")]/mfinal_df.loc[:,"Total Votes"]).astype(float).round(4)*100
mfinal_df.loc[:,(ogwin,"%")]=(mfinal_df.loc[:,(ogwin,"Votes")]/mfinal_df.loc[:,"Total Votes"]).astype(float).round(4)*100
mfinal_df.loc[:,("Margin","%")]=(mfinal_df.loc[:,("Margin","Votes")]/mfinal_df.loc[:,"Total Votes"]).astype(float).round(4)*100

mfinal_df

States Trump/Vance Republican            Harris/Walz Democratic  \
                                    Votes      %  EV                  Votes   
0       California                6081697   38.9   0                9043413   
1         New York                3578899  43.31   0                4619195   
2    Massachusetts                1251303  36.02   0                2126518   
3         Maryland                1035550  34.08   0                1902577   
4       Washington                1530923  39.01   0                2245849   
5         Illinois                2449079  43.47   0                3062863   
6         Colorado                1377441  43.14   0                1728159   
7           Oregon                 919480  40.97   0                1240600   
8             D.C.                  21076   6.47   0                 294185   
9         Virginia                2075085  46.05   0                2335395   
10     Connecticut                 736918  41.89   0                 992053   
11      New Jersey                1968215  46.06   0                2220713   
12       Minnesota                1519032  46.68   0                1656979   
13          Hawaii                 193661  37.48   0                 313044   
14         Vermont                 119395  32.32   0                 235791   
15            ME-1                 165214  38.09   0                 258863   
16        Delaware                 214351  41.79   0                 289758   
17    Rhode Island                 214406  41.76   0                 285156   
18           Maine                 377977  45.46   0                 435652   
19      New Mexico                 423391  45.85   0                 478802   
20   New Hampshire                 395523  47.87   0                 418488   
21            NE-2                 148905  46.73   0                 163541   
0        Wisconsin                1697626  49.16   0                1698626   
2     Pennsylvania                3543308  49.35   0                3544308   
1         Michigan                2816636  49.03   0                2817636   
10            ME-2                 212763   53.5   1                 176789   
15            NE-1                 177666  55.49   1                 136153   
6           Alaska                 184458  54.54   3                 140026   
4           Nevada                 751205  50.59   6                 705197   
3          Georgia                2663117  50.72  16                2548017   
8          Montana                 352079  58.39   4                 231906   
14         Wyoming                 192633   71.6   3                  69527   
16    South Dakota                 272081  63.43   3                 146859   
17    North Dakota                 246505  66.96   3                 112327   
33            NE-3                 238245  76.03   1                  70301   
5   North Carolina                2898423  50.86  16                2715375   
7          Arizona                1770242  52.22  11                1582860   
32        Nebraska                 564816  59.32   2                 369995   
9           Kansas                 758802  57.16   6                 544853   
11            Iowa                 927019  55.73   6                 707278   
18     Mississippi                 747744  60.89   6                 466668   
29   West Virginia                 533556  69.97   4                 214309   
22            Utah                 883818  59.38   6                 562566   
30           Idaho                 605246  66.87   4                 274972   
25        Arkansas                 759241   64.2   6                 396905   
24       Louisiana                1208505  60.22   8                 766870   
20  South Carolina                1483747  58.23   9                1028452   
26        Oklahoma                1036213  66.16   7                 499599   
23        Missouri                1751986  58.49  10          

# Overflow: Multiple Source States

In [73]:
ow_df=losers_df.copy(deep=True)

ol_df=winners_df.copy(deep=True)

sources=3 #number of source states
flipped_EV=0
total_voters_moved=0 #the number of voters moved overall
state_voters_moved=0 #the number of voters moved into each state to flip it
mini_margin=1000  #our arbitrary minimum acceptable margin of statewide victory

i=0 # source state index
j=0 # target state index
pover=0 # if a state's overflow is less than the margin, carry over to the next source state's overflow

source_state=ow_df.iloc[i] #the state we are moving voters from
target_state=ol_df.iloc[j] #the state we are moving voters to
overflow=source_state.loc[("Margin","#")]*(-1)-mini_margin+pover # the number of voters we have to move from the source state, plus any leftover voters we had to carry over from the previous source state


while i<sources and j<len(ol_df): # while we still have source states and target states to move voters from/to

  if overflow>(target_state.loc[("Margin","#")]+mini_margin): #if we have enough voters in the source state to flip the target state with a mini_margin, move the number of voters needed to flip the target state with a mini_margin
    pover=0 # reset pover since we have enough voters to flip the target state
    
    state_voters_moved=target_state.loc[("Margin","#")]+mini_margin
    # change votes
    ol_df.loc[j,(oglos,"#")]=int(ol_df.loc[j,(oglos,"#")])+state_voters_moved
    ol_df.loc[j,"Total Votes"]=int(ol_df["Total Votes"].iloc[j])+state_voters_moved
    ol_df.loc[j,("Margin","#")]=mini_margin*(-1)

    # change EV
    flipped_EV=flipped_EV+ol_df.loc[j,(ogwin,"EV")]
    ol_df.loc[j,(oglos,"EV")]=ol_df.loc[j,(ogwin,"EV")]
    ol_df.loc[j,(ogwin,"EV")]=0
    
    # change overflow
    overflow=overflow-state_voters_moved
    total_voters_moved=total_voters_moved+state_voters_moved # update total voters moved
    
    print(f"Flipped {target_state.loc['States'].to_string()[4:]} by moving {state_voters_moved} voters from {source_state.loc['States'].to_string()[4:]}. Total voters moved: {total_voters_moved}. Flipped EV: {flipped_EV}. Overflow: {overflow}.")

    
    # move on to the next target state
    j=j+1
    target_state=ol_df.iloc[j] 

  else: # store overflow in pover and move on to the next source state
    print(f"Not enough voters in {source_state.loc['States'].to_string()[4:]} to flip {target_state.loc['States'].to_string()[4:]}. Overflow: {overflow}. Moving on to the next source state.")
    pover=overflow
    i+=1 
    source_state=ow_df.iloc[i] if i<sources else None
    overflow=source_state.loc[("Margin","#")]*(-1)-mini_margin+pover if i<sources else None # update overflow for the new source state




# ol_df

Flipped Florida by moving 75309 voters from Texas. Total voters moved: 75309. Flipped EV: 29. Overflow: 1185410.
Flipped Ohio by moving 167272 voters from Texas. Total voters moved: 242581. Flipped EV: 47. Overflow: 1018138.
Flipped New Hampshire by moving 40643 voters from Texas. Total voters moved: 283224. Flipped EV: 51. Overflow: 977495.
Flipped Nevada by moving 68806 voters from Texas. Total voters moved: 352030. Flipped EV: 57. Overflow: 908689.
Flipped Virginia by moving 150298 voters from Texas. Total voters moved: 502328. Flipped EV: 70. Overflow: 758391.
Flipped Colorado by moving 138858 voters from Texas. Total voters moved: 641186. Flipped EV: 79. Overflow: 619533.
Flipped Iowa by moving 92927 voters from Texas. Total voters moved: 734113. Flipped EV: 85. Overflow: 526606.
Flipped Pennsylvania by moving 310840 voters from Texas. Total voters moved: 1044953. Flipped EV: 105. Overflow: 215766.
Flipped New Mexico by moving 80547 voters from Texas. Total voters moved: 1125500. 

In [74]:
# update the source states
ow_df=losers_df.copy(deep=True)

i=0
reimburse = total_voters_moved
while i<sources:
    if reimburse>ow_df.loc[i,("Margin","#")]*(-1)-mini_margin:
        state_voters_removed=ow_df.loc[i,("Margin","#")]*(-1)-mini_margin

        # change votes
        ow_df.loc[i,(oglos,"#")]=int(ow_df.loc[i,(oglos,"#")])-state_voters_removed
        ow_df.loc[i,"Total Votes"]=int(ow_df["Total Votes"].iloc[i])-state_voters_removed
        ow_df.loc[i,("Margin","#")]=mini_margin*(-1)

        reimburse-=state_voters_removed
        print(f"Removed {state_voters_removed} voters from {ow_df.loc[i,'States'].to_string()[4:]}. Remaining voters to reimburse: {reimburse}.")
    else:
        state_voters_removed=reimburse

        # change votes
        ow_df.loc[i,(oglos,"#")]=int(ow_df.loc[i,(oglos,"#")])-state_voters_removed
        ow_df.loc[i,"Total Votes"]=int(ow_df["Total Votes"].iloc[i])-state_voters_removed
        ow_df.loc[i,("Margin","#")]=ow_df.loc[i,("Margin","#")]+state_voters_removed

        reimburse-=state_voters_removed
        print(f"Removed {state_voters_removed} voters from {ow_df.loc[i,'States'].to_string()[4:]}. All voters reimbursed.")
    i+=1
        
# ow_df

Removed 1260719 voters from Texas. Remaining voters to reimburse: 987411.
Removed 500621 voters from Tennessee. Remaining voters to reimburse: 486790.
Removed 486790 voters from Utah. All voters reimbursed.


In [76]:
# concatenate the winner and loser
final_df=pd.concat([ol_df,ow_df]).sort_values(by=[("Margin","#")])

#update percentages
final_df.loc[:,(oglos,"%")]=(final_df.loc[:,(oglos,"#")]/final_df.loc[:,"Total Votes"]).astype(float).round(4)*100
final_df.loc[:,(ogwin,"%")]=(final_df.loc[:,(ogwin,"#")]/final_df.loc[:,"Total Votes"]).astype(float).round(4)*100
final_df.loc[:,("Margin","%")]=(final_df.loc[:,("Margin","#")]/final_df.loc[:,"Total Votes"]).astype(float).round(4)*100

final_df

States Obama/Biden Democratic             \
                                              #      %  EV   
3                Alabama                 795696  38.36   0   
4               Oklahoma                 443547  33.23   0   
5               Kentucky                 679370   37.8   0   
6              Louisiana                 809141  40.58   0   
7                Georgia                1773827  45.48   0   
8                Indiana                1152887  43.93   0   
9               Missouri                1223796  44.38   0   
10              Arkansas                 394409  36.88   0   
11                Kansas                 440726  37.99   0   
12               Arizona                1025232  44.59   0   
13                 Idaho                 212787   32.4   0   
14        South Carolina                 865941  44.09   0   
15         West Virginia                 238269  35.54   0   
16              Nebraska                 302081  38.03   0   
17           Mississippi                 562949  43.79   0   
18                  NE-3                  72110  27.82   0   
19               Wyoming                  69286  27.82   0   
20        North Carolina                2178391  48.35   0   
21               Montana                 201839   41.7   0   
22          South Dakota                 145039  39.87   0   
23          North Dakota                 124827  38.69   0   
24                  NE-1                 108082  40.83   0   
25                Alaska                 122640  40.81   0   
26                  NE-2                 121889   45.7   0   
2                   Utah                 251813  47.45   0   
1              Tennessee                 960709  49.07   0   
0                  Texas                3308124  49.13   0   
0                Florida                4237756  49.57   0   
13                  ME-2                 177998  48.63   0   
1                   Ohio                2827709  49.19   0   
2          New Hampshire                 369561  49.17   0   
3                 Nevada                 531373  49.03   0   
4               Virginia                1971820  49.24   0   
5               Colorado                1323102  48.85   0   
6                   Iowa                 822544   49.1   0   
7           Pennsylvania                2990274  49.31   0   
8             New Mexico                 415335  48.05   0   
9              Wisconsin                1620985  49.38   0   
10             Minnesota                1546167  48.88   0   
11              Delaware                 242584   49.3   0   
14          Rhode Island                 279677  49.11   0   
12              Michigan                2564569   49.5   0   
27                  ME-1                 223035  59.57   1   
16               Vermont                 199239  66.57   3   
22                 Maine                 401306  56.27   2   
21                Hawaii                 306658  70.55   4   
15                Oregon                 970488  54.24   7   
28  District of Columbia                 267070  90.91   3   
17           Connecticut                 905083  58.06   7   
18            Washington                1755396  56.16  12   
20            New Jersey                2125101  58.38  14   
26              Maryland                1677844  61.97  10   
24         Massachusetts                1921290  60.65  11   
19              Illinois                3019512   57.6  20   
25              New York                4485741  63.35  29   
23            California                7854285  60.24  55   

   Romney/Ryan Republican              Margin        Total Votes        VPP  
                        #      %  EV        #      %                         
3                 1255925  60.55   9  -460229 -22.19     2074338  -51136.56  
4                  891325  66.77   7  -447778 -33.54     1334872  -63968.29  
5                 1087190  60.49   8  -407820 -22.69     1797212  -50977.50  
6                 1152262  57.7